In [3]:
import os
import pandas as pd
import numpy as np

def extract_axis_features_from_folder(folder_path):
    """
    Iterates through a folder of CSV files.
    Extracts Mean, Std, Min, Max for the first 6 columns (sensor data).
    Returns a DataFrame where every row is a file's 24-feature vector.
    """
    
    features_list = []
    file_names = []
    
    # Define column names for clarity. ADJUST THESE if your CSV headers are different.
    # We assume the order is: Acc_X, Acc_Y, Acc_Z, Gyro_X, Gyro_Y, Gyro_Z 
    # (after Time/Timestamp column, if present).
    col_labels = ['Acc_X', 'Acc_Y', 'Acc_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z']
    
    print(f"Starting feature extraction from folder: {folder_path}")
    
    # Loop through every file in the directory
    for file in os.listdir(folder_path):
        if file.endswith(".csv"):
            file_path = os.path.join(folder_path, file)
            
            try:
                # Load the CSV file
                # Phyphox CSVs often have a 'Time (s)' column first.
                df = pd.read_csv(file_path, sep=';')
                
                # Check column count and skip the first (Time) column
                if df.shape[1] >= 7:
                    # Select 6 sensor columns (from index 1 to 6, excluding Time at index 0)
                    data = df.iloc[:, 1:7]
                elif df.shape[1] == 6:
                    # Select all 6 columns if no Time column is present
                    data = df.iloc[:, :6]
                else:
                    print(f"Skipping {file}: Not enough columns found ({df.shape[1]} columns).")
                    continue
                
                if data.empty:
                    print(f"Skipping {file}: Data frame is empty after selection.")
                    continue

                # --- FEATURE CALCULATION ---
                # Calculate the 4 required statistics for the 6 axes
                means = data.mean().values
                stds = data.std().values
                mins = data.min().values
                maxs = data.max().values
                
                # Concatenate them into one long feature vector (4 stats * 6 axes = 24 features)
                # Order: [Mean_X...Mean_Z, Std_X...Std_Z, Min_X...Min_Z, Max_X...Max_Z]
                feature_vector = np.concatenate([means, stds, mins, maxs])
                
                features_list.append(feature_vector)
                file_names.append(file)
                
            except Exception as e:
                print(f"Error processing {file}: {e}")

    # Create the column headers for the final DataFrame
    stats = ['Mean', 'Std', 'Min', 'Max']
    final_headers = [f"{stat}_{col}" for stat in stats for col in col_labels]
    
    # Create a Pandas DataFrame for the results
    result_df = pd.DataFrame(features_list, columns=final_headers, index=file_names)
    
    return result_df

# --- USAGE SECTION ---

# 1. CHANGE THIS PATH to the folder containing your Phyphox CSV files
my_folder_path = './Human activity/Juming'

if os.path.exists(my_folder_path):
    # 2. Run the function
    features_df = extract_axis_features_from_folder(my_folder_path)
    
    # Print the result (First 5 rows)
    print("\nFeatures extracted successfully:")
    print(features_df.head())
    
    # 3. Optional: Save the features to a new CSV file ready for training
    output_file_name = 'extracted_HAR_features_24D.csv'
    features_df.to_csv(output_file_name)
    print(f"\nFeatures saved to {output_file_name}")
else:
    print(f"ERROR: The specified folder path does not exist: {my_folder_path}")

Starting feature extraction from folder: ./Human activity/Juming
Error processing Jumping Haakon.csv: Could not convert ['0,2418143751,0199298860,998382092-1,017535686-1,422155857-0,1699883190,0885854660,775721371-3,045424461-36,28292847-60,81691366,086359978-4,034229755-6,158185959-26,78243828-12,65604973-11,68639755-14,94999409-13,27614784-13,23065758-14,58577633-13,50838566-13,69034481-15,25884628-11,47091961-4,678270342,1452047824,022258759-7,925107002-5,042488098-1,6041151292,4756045345,8038439754,809951305-0,316034615-2,23139596-2,499546528-1,769315004-0,945709646-0,940921247-1,053448796-1,000776291-1,175553083-0,93134445-0,524330199-0,385466486-0,5674257872,5019407273,679888012,604891539-9,598653793-73,63845825-10,872368814,2616791735,992986202-23,68404198-12,61534786-13,62570095-13,78371811-14,54028606-11,8085022-14,51873875-13,17319679-13,88188076-11,66724396-4,0462007524,999392509-6,632237911-22,68326569-10,271424291,9632453926,9434843067,6641387943,050212860,679953277-1,5251